In [ ]:
# Example 1: verify the Jupyter server extension entry points exist and have the expected shape.
import jupyterlab_commands_toolkit as jct

lab_paths = jct._jupyter_labextension_paths()
assert isinstance(lab_paths, list) and lab_paths, lab_paths
assert lab_paths[0]["dest"] == "jupyterlab-commands-toolkit", lab_paths
assert lab_paths[0]["src"] == "labextension", lab_paths

server_points = jct._jupyter_server_extension_points()
assert isinstance(server_points, list) and server_points, server_points
assert server_points[0]["module"] == "jupyterlab_commands_toolkit", server_points

print("labextension paths:", lab_paths)
print("server extension points:", server_points)

In [ ]:
# Example 2: inspect the tools submodule — confirm the MCP-discoverable TOOLS list and async API surface.
import inspect

from jupyterlab_commands_toolkit import tools

assert hasattr(tools, "TOOLS"), "tools module must expose TOOLS list for jupyter-server-mcp"
assert isinstance(tools.TOOLS, list) and len(tools.TOOLS) >= 2, tools.TOOLS
for entry in tools.TOOLS:
    assert entry.startswith("jupyterlab_commands_toolkit.tools:"), entry

# Both primary callables are coroutine functions.
assert inspect.iscoroutinefunction(tools.list_all_commands)
assert inspect.iscoroutinefunction(tools.execute_command)
assert inspect.iscoroutinefunction(tools.emit_and_wait_for_result)

print("TOOLS:", tools.TOOLS)
print("list_all_commands signature:", inspect.signature(tools.list_all_commands))
print("execute_command signature:", inspect.signature(tools.execute_command))

In [ ]:
# Example 3: pending_requests storage starts empty and handle_command_result is a no-op for unknown ids.
from jupyterlab_commands_toolkit import tools

# Reset to a known state in case prior cells touched it.
tools.pending_requests.clear()
assert tools.pending_requests == {}

# Feeding an unknown requestId must not raise and must not insert anything.
tools.handle_command_result({"requestId": "does-not-exist", "success": True, "result": 42})
assert tools.pending_requests == {}, tools.pending_requests

# Missing requestId also must not raise.
tools.handle_command_result({"success": False, "error": "no id"})
assert tools.pending_requests == {}, tools.pending_requests

print("handle_command_result is robust to unknown / missing requestIds")

In [ ]:
# Example 4: handle_command_result completes a registered future end-to-end.
import asyncio

from jupyterlab_commands_toolkit import tools

tools.pending_requests.clear()

loop = asyncio.get_event_loop()
fut = loop.create_future()
tools.pending_requests["req-1"] = {
    "timestamp": 0.0,
    "data": {"name": "noop", "args": {}},
    "result": None,
    "completed": False,
    "future": fut,
}

payload = {"requestId": "req-1", "success": True, "result": {"value": 99}}
tools.handle_command_result(payload)

assert tools.pending_requests["req-1"]["completed"] is True
assert tools.pending_requests["req-1"]["result"] == payload
assert fut.done() and fut.result() == payload

tools.pending_requests.clear()
print("handle_command_result resolves the registered future with the payload")

In [ ]:
# Example 5: _load_jupyter_server_extension exists and is callable; module declares __version__.
import jupyterlab_commands_toolkit as jct

assert callable(jct._load_jupyter_server_extension), "server extension loader must be callable"
assert isinstance(jct.__version__, str) and jct.__version__, jct.__version__

# The schema files the loader registers must ship inside the wheel.
import pathlib
events_dir = pathlib.Path(jct.__file__).parent / "events"
assert events_dir.is_dir(), f"events dir missing: {events_dir}"
cmd_schema = events_dir / "jupyterlab-command.yml"
res_schema = events_dir / "jupyterlab-command-result.yml"
assert cmd_schema.is_file(), cmd_schema
assert res_schema.is_file(), res_schema

print(f"version={jct.__version__}")
print(f"event schemas present: {cmd_schema.name}, {res_schema.name}")
assert True